In [ ]:
#Installing the necessary libraries

In [ ]:
%%capture
#!pip -q install geopandas
#!pip -q install geojson
!pip -q install rasterio
#!pip -q install geemap
#!pip -q install eeconvert

#Panel interactively
!pip install jupyter_bokeh

In [ ]:
# Standard imports
import os
#from tqdm.auto import tqdm
#import requests
#import json

import pandas as pd
import numpy as np
from PIL import Image

# Geospacial processing packages
import geopandas as gpd
#import geojson

#import shapely
import rasterio as rio
#from rasterio.plot import show
import rasterio.mask
from shapely.geometry import box

# Mapping and plotting libraries
import matplotlib.pyplot as plt
import matplotlib.colors as cl
#import ee
#import eeconvert as eec
#import geemap
#import geemap.eefolium as emap # Commented out the old import
#import geemap.folium as emap # Added the new import
import folium

# Model
import torch
from torchvision import datasets, models, transforms

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
import panel as pn
pn.extension()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Upload to Tiles

## Create tiles

In [ ]:
def generate_tiles(image_file, size=64):
    """Genarates size*size polygon tiles.
    """

    # Open the raser image using rasterio
    raster = rio.open(image_file)
    width, height = raster.shape

    # Create a dictionary which will conatin our 64Z64 px polygon tiles
    # Later convert this dict into GeoPandas DataFrame
    geo_dict = {"id":[],"geometry":[]}
    index = 0

    # Do a sliding windows across the raste image
    for w in range(0, width, size):
      for h in range(0, height, size):
          # Create a Window of your disired size
          window = rio.windows.Window(h, w, size, size)

          # Get the georeferenced windowss bounds
          bbox = rio.windows.bounds(window, raster.transform)

          # Create a shapely geometry from the bounding box
          bbox = box(*bbox)

          # Create a unique id for each geometry
          uid = str(index)

          # Update dictionary
          geo_dict["id"].append(uid)
          geo_dict["geometry"].append(bbox)

          index += 1

    # Cast dictionary as a GeoPandas DataFrame
    results = gpd.GeoDataFrame(pd.DataFrame(geo_dict))

    # Set CRS to EPSG: 4326
    results.set_crs("EPSG:4326", inplace=True)

    raster.close()

    return results

## Predict

In [ ]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def predict_crop(image, shape, classes, model, show=False):
    """
    Generate prediciton
    """
    with rio.open(image) as src:
        # Crop source image using polyhon shape
        ## https://rasterio.readthedocs.io/en/latest/api/rasterio.mask.html#rasterio.mask.mask

        out_image, out_transform = rio.mask.mask(src, shape, crop=True)

        # Crop out black (zero) border
        _, x_nonzero, y_nonzero = np.nonzero(out_image)
        out_image = out_image[
            :,
            np.min(x_nonzero):np.max(x_nonzero),
            np.min(y_nonzero):np.max(y_nonzero)
        ]

        # Get the metadata of the source image and update it
        # with the width, height, and transfor, of the cropped image
        out_meta = src.meta
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

        # Save the cropped image as a temporatry TIFF file.
        temp_tif = "temp.tif"
        with rio.open(temp_tif, "w", **out_meta) as dest:
            dest.write(out_image)

        # Open the cropped image and genrated prediction
        # Using the model
        image = Image.open(temp_tif)
        input = transform(image)
        input = input.to(device)
        output = model(input.unsqueeze(0))
        _, pred = torch.max(output, 1)
        label = str(classes[int(pred[0])])

        if show:
            out_image.show(title=label)

        return label

    return None

## Final Function

In [ ]:
import io
# File input widget
file_input = pn.widgets.FileInput(accept='.tif')

message = pn.widgets.StaticText(name='Message', value='Nothing')

# Get file when imputed
def update_map(input_file=None):
  if input_file:
    message.value = "Uploaded"
    # Convert the uploaded binary data to a file-like object
    file_bytes = io.BytesIO(file_input.value)
    # Open with rasterio
    image = rio.open(file_bytes)

    message.value = "Creating tiles"
    tiles = generate_tiles(file_bytes, size=64)

    # LULC Classes
    classes = [
        "AnnualCrop",
        "Forest",
        "HerbaceousVegetation",
        "Highway",
        "Industrial",
        "Pasture",
        "PermanentCrop",
        "Residential",
        "River",
        "SeaLake"
    ]

    message.value = "Loading model"
    #Get the model
    path_drive = "./drive/MyDrive/Colab Notebooks/Comp702/Models/Retrained/RESNET50_0001_SDG-RGB_B32_WW_EP150_S224.pth"

    model_2 = models.resnet50()
    model_2.fc = torch.nn.Linear(in_features=model_2.fc.in_features, out_features=len(classes))
    model_2.load_state_dict(torch.load(f=path_drive, map_location=device))
    model_2 = model_2.to(device)

    model_2.eval()



    # Get label

    #
    message.value = "Making prediction: "
    labels = [] # Stor prediction
    #for index in tqdm(range(len(tiles)), total=len(tiles)):
    l = len(tiles)
    for index in range(l):
      message.value = f"Making prediction: {index}/{l}"
      try:
        # Use model_2 instead of model
        label = predict_crop(file_bytes, [tiles.iloc[index]['geometry']], classes, model_2)
        labels.append(label)
      except:
        labels.append("Error")
        continue

    # Create tile
    tiles['pred'] = labels


    message.value = "Creating map..."

    # We map each class to a corresponding color
    colors = {
      'AnnualCrop' : 'lightgreen',
      'Forest' : 'forestgreen',
      'HerbaceousVegetation' : 'yellowgreen',
      'Highway' : 'gray',
      'Industrial' : 'red',
      'Pasture' : 'mediumseagreen',
      'PermanentCrop' : 'chartreuse',
      'Residential' : 'magenta',
      'River' : 'dodgerblue',
      'SeaLake' : 'blue',
      'Error' : 'black'
    }
    tiles['color'] = tiles["pred"].apply(
      lambda x: cl.to_hex(colors.get(x))
    )
    # Divide Tiles





    # Instantiate map centered on the centroid
    map = folium.Map(zoom_start=10)


    # Add Google Satellite basemap
    folium.TileLayer(
          tiles = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
          attr = 'Google',
          name = 'Google Satellite',
          overlay = True,
          control = True
    ).add_to(map)

    # Add LULC Map with legend
    legend_txt = '<span style="color: {col};">{txt}</span>'
    for label, color in colors.items():

      # Specify the legend color
      name = legend_txt.format(txt=label, col=color)
      feat_group = folium.FeatureGroup(name=name)

      # Add GeoJSON to feature group
      subtiles = tiles[tiles.pred==label]
      if len(subtiles) > 0:
        folium.GeoJson(
            subtiles,
            style_function=lambda feature: {
              'fillColor': feature['properties']['color'],
              'color': 'black',
              'weight': 0,
              'fillOpacity': 0.5,
            },
            name='LULC Map'
        ).add_to(feat_group)
        map.add_child(feat_group)

    folium.LayerControl().add_to(map)
    # Reset zoom with data available
    map.fit_bounds(map.get_bounds())
    message.value = "Done!"
    return pn.pane.HTML(map._repr_html_(), width=900, height=900)
    # Classify
    # Process

    # Return map
"""
"""
# Attach callback
#file_input.param.watch(update_plot, 'value')
mapa= pn.bind(update_map, file_input)
# Layout
main_area=pn.Column(
    "# Upload GeoTIFF and View",
    file_input,
    mapa,
    message
)

In [ ]:
MAIN=pn.WidgetBox(main_area)

In [ ]:
tab2 = pn.GridSpec(width=1100, height=700, nrows=3, ncols=3)

tab2[0:3, 0:3] = MAIN  # Filters

tab2

GridSpec(height=700, ncols=3, nrows=3, sizing_mode='fixed', width=1100)
    [0] WidgetBox(height=700, width=1100)
        [0] Column
            [0] Markdown(str)
            [1] FileInput(accept='.tif')
            [2] ParamFunction(function, _pane=Str, defer_load=False)
            [3] StaticText(name='Message', value='Nothing')